# 02 — Data Quality

Pergunta: **onde estao ausencias, duplicatas e divergencias de label capazes de invalidar a avaliacao?** Nenhuma linha e removida.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, ensure_output_directories
from src.data.make_dataset import build_interim_datasets, load_interim_datasets

cfg = load_config()
ensure_output_directories(cfg)
if not list(Path(cfg["data"]["interim"]).glob("*.parquet")):
    build_interim_datasets()
datasets = {str(frame["dataset"].iloc[0]): frame for frame in load_interim_datasets().values()}
list(datasets)


In [ ]:
from src.analysis.profiling import missingness_table, label_distribution
from src.analysis.duplicates import duplicate_summary, cross_dataset_duplicates, near_duplicate_candidates

missing = pd.concat([missingness_table(frame, name) for name, frame in datasets.items()], ignore_index=True)
missing.sort_values(["dataset", "missing_percent"], ascending=[True, False])

## Duplicatas completas, de titulo, texto e URL

In [ ]:
duplicates = pd.concat([duplicate_summary(frame, name) for name, frame in datasets.items()], ignore_index=True)
duplicates

## Sobreposicao entre datasets

`not_fake` e `real` sao equiparados somente para marcar conflitos; labels originais nao sao alterados.

In [ ]:
cross = cross_dataset_duplicates(datasets)
display(cross.groupby(["match_type", "datasets", "label_conflict"]).size().rename("groups").reset_index())
display(cross.head(20))

## Candidatos quase duplicados

TF-IDF de n-gramas de caracteres com similaridade >= 0,96 gera candidatos para revisao, nao exclusoes automaticas.

In [ ]:
near = []
for name, frame in datasets.items():
    if frame["text"].fillna("").str.len().ge(80).sum() >= 2:
        near.append({"dataset": name, "candidate_pairs": len(near_duplicate_candidates(frame, threshold=0.96))})
pd.DataFrame(near)

## Definicoes e distribuicoes dos labels

Nao ha unificacao automatica. `other` permanece uma terceira semantica.

In [ ]:
pd.concat([label_distribution(frame, name) for name, frame in datasets.items()], ignore_index=True)